# 00 — Exploratory Data Analysis

IEEE-CIS Fraud Detection dataset. Understand class imbalance, null rates, amount distribution, and identity coverage before feature engineering.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../../..") / "data" / "ieee-fraud-detection"


## 1. Load raw data

In [ ]:
txn = pd.read_csv(DATA_DIR / "train_transaction.csv")
idn = pd.read_csv(DATA_DIR / "train_identity.csv")
print(f"Transactions : {len(txn):,} rows  ×  {txn.shape[1]} columns")
print(f"Identity     : {len(idn):,} rows  ×  {idn.shape[1]} columns")


## 2. Class imbalance

In [ ]:
fraud_rate = txn["isFraud"].mean()
print(f"Fraud rate : {fraud_rate:.2%}")
print(f"Fraud count: {txn['isFraud'].sum():,}  /  {len(txn):,} total")
txn["isFraud"].value_counts().rename({0: "Legit", 1: "Fraud"})


## 3. Transaction amount statistics

In [ ]:
print(f"Mean   : ${txn['TransactionAmt'].mean():.2f}")
print(f"Median : ${txn['TransactionAmt'].median():.2f}")
print()
print("Percentiles:")
print(txn["TransactionAmt"].quantile([0.25, 0.5, 0.75, 0.90, 0.95, 0.99]).to_string())


## 4. Fraud vs legitimate — amount comparison

In [ ]:
txn.groupby("isFraud")["TransactionAmt"].describe().round(2)


## 5. Identity join coverage

In [ ]:
joined = txn.merge(idn, on="TransactionID", how="left", indicator=True)
coverage = (joined["_merge"] == "both").mean()
print(f"Transactions with identity record: {coverage:.1%}")
print(f"  → missing identity is itself a fraud signal")


## 6. Null rates — top 20 columns

In [ ]:
null_rates = txn.isnull().mean().sort_values(ascending=False).head(20)
null_rates[null_rates > 0].rename("null_rate").to_frame().style.format("{:.1%}")


## 7. Key feature null rates

In [ ]:
key_cols = ["dist1", "dist2", "R_emaildomain", "P_emaildomain"]
for col in key_cols:
    if col in txn.columns:
        print(f"{col:20s}: {txn[col].isnull().mean():.1%} null")


## 8. Product and card distributions

In [ ]:
print("=== ProductCD ===")
print(txn["ProductCD"].value_counts().to_string())
print()
if "card4" in txn.columns:
    print("=== card4 (brand) ===")
    print(txn["card4"].value_counts().to_string())
